In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Polygon

import sys
import os

# Remplacez par le chemin absolu vers votre dossier fusion_zada_app
project_path = r"D:\Dev\Projets\fusion_zada_app"
if project_path not in sys.path:
    sys.path.append(project_path)

from app.modules.nlp import NLPEngine

# 1) Jeu de données jouet
geoms = [
    Polygon([(0,0),(1,0),(1,1),(0,1)]),
    Polygon([(1,0),(2,0),(2,1),(1,1)]),
    Polygon([(0,1),(1,1),(1,2),(0,2)]),
]
data = pd.DataFrame({
    "desc": [
        "faune: elephant; primate: singe; habitat: foret tropicale",
        "agriculture: manioc; habitat: savane arbustive",
        "faune: buffle; habitat: foret humide; primate: gorille"
    ],
    "source": ["A","B","C"]
})
gdf = gpd.GeoDataFrame(data, geometry=geoms, crs="EPSG:4326")



In [3]:
# 2) Mock: le builder va concaténer toutes les colonnes non exclues -> OK
#    (si ton build_corpus_from_fusion_gdf exclut 'source', ça ne gêne pas)

# 3) Instancier et initialiser
nlp = NLPEngine()
# Optionnel: forcer backend si tu veux n'utiliser que Word2Vec local par ex.
# nlp.set_backend("word2vec")
nlp.init_from_fusion_gdf(gdf)

# 4) RECHERCHE SEMANTIQUE
df_sem = nlp.search("foret tropicale primate", top_k=5, mode="semantic")
gj_sem, legend_sem, bounds_sem = nlp.to_geojson(df_sem)
print("=== SEMANTIC ===")
print(df_sem[["id_zone","score","similarite","couverture","mode"]].head())
print("Legend (semantic):", legend_sem["items"][0])

=== SEMANTIC ===
   id_zone     score  similarite  couverture      mode
0        0  0.918822    0.918822    1.000000  semantic
1        2  0.894235    0.894235    0.666667  semantic
2        1  0.666571    0.666571    0.000000  semantic
Legend (semantic): {'label': '0.67 - 0.74', 'color': '#313695'}


In [4]:
# 5) RECHERCHE MOTS-CLÉS (AND) — 100% si tous présents
df_kw = nlp.search("elephant singe", top_k=5, mode="keyword")
gj_kw, legend_kw, bounds_kw = nlp.to_geojson(df_kw)
print("\n=== KEYWORD (AND) ===")
print(df_kw[["id_zone","score","similarite","couverture","mode"]].head())


=== KEYWORD (AND) ===
   id_zone  score  similarite  couverture     mode
0        0    1.0         0.0         1.0  keyword
1        2    0.0         0.0         0.0  keyword
2        1    0.0         0.0         0.0  keyword


In [5]:
# 6) Assertions légères
assert (df_kw["score"].max() <= 1.0) and (df_kw["score"].min() >= 0.0)
assert gj_sem["type"] == "FeatureCollection" and gj_kw["type"] == "FeatureCollection"
print("\nSmoke test OK ")


Smoke test OK 
